# NimbusPay ticket triage: fine-tuning Qwen2.5-1.5B-Instruct with LoRA

**Pipeline:** audit & clean data → base-model baselines → train final adapter → dev evaluation → error analysis
→ test predictions (+ reproducibility check) → experiment table.

**Final model:** `Qwen/Qwen2.5-1.5B-Instruct`, loaded in **fp16 (not 4-bit)**, LoRA r=16 / alpha=32 on all 7 linear
projections, trained on the cleaned training set + 400 threshold-focused augmented rows (experiment E4).

**Runs top to bottom on a fresh T4 (~1h15m).** Colab: put `training_assessment.zip`, `audit_clean.py`, `augment.py`
(and the experiment logs `experiments.jsonl`, `experiments_kaggle.jsonl`) in `MyDrive/vima3ya/`.
Kaggle: add them as an input dataset, Accelerator = GPU T4, Internet on.

## 0. Platform, one T4, pinned libraries

In [1]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Kaggle offers 2x T4; all runs use exactly one T4
os.environ["TOKENIZERS_PARALLELISM"] = "false"
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
IN_KAGGLE = os.path.exists("/kaggle")
print("platform:", "Colab" if IN_COLAB and not IN_KAGGLE else "Kaggle" if IN_KAGGLE else "other")

platform: Kaggle


In [2]:
!pip -q install "transformers==5.22.2" "peft==0.21.2" "trl==1.17.1" "bitsandbytes==0.50.2" "accelerate==1.15.0" "datasets==5.2.0"
# Colab ships torchao 0.10; PEFT refuses torchao < 0.16. torchao is not used, so remove it.
!pip -q uninstall -y torchao
import importlib, torch
VERSIONS = {m: importlib.import_module(m).__version__ for m in
            ["torch", "transformers", "peft", "trl", "bitsandbytes", "accelerate", "datasets"]}
print(VERSIONS)
print(torch.cuda.get_device_name(0), "| GPUs visible:", torch.cuda.device_count(), "| bf16:", torch.cuda.is_bf16_supported())
assert torch.cuda.device_count() == 1 and "T4" in torch.cuda.get_device_name(0)

{'torch': '2.14.1+cu130', 'transformers': '5.22.2', 'peft': '0.21.2', 'trl': '1.17.1', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0', 'datasets': '5.2.0'}
Tesla T4 | GPUs visible: 1 | bf16: True


## 1. Inputs, data audit and cleaning
`audit_clean.py` applies every fix/drop decision and writes `cleaning_log.csv`; it then re-derives every rule-checkable field on the cleaned data (expected: `verification … none`). dev/test are never modified.

In [3]:

import os, glob, shutil, zipfile, json, time, math, gc
import subprocess, collections, random, re
import numpy as np
import pandas as pd

# Detect the environment
IN_KAGGLE = os.path.exists("/kaggle/input")


# FORCE GOOGLE COLAB ENVIRONMENT

IN_KAGGLE = False

from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/vima3ya"
PROJ = SRC

print("Environment: Google Colab")
print("Source folder:", SRC)
print("Project folder:", PROJ)

# Verify required files
required = [
    "training_assessment.zip",
    "audit_clean.py",
    "augment.py",
    "experiments.jsonl",
    "experiments_kaggle.jsonl"
]

for f in required:
    path = os.path.join(SRC, f)
    assert os.path.isfile(path), f"Missing required file: {path}"
    print("FOUND:", f)


WORK = "/tmp/work"
OUT = f"{PROJ}/final"

for d in (WORK, OUT, f"{OUT}/adapter"):
    os.makedirs(d, exist_ok=True)

def find(name):
    path = os.path.join(SRC, name)
    return path if os.path.exists(path) else None

# Extract training dataset
with zipfile.ZipFile(find("training_assessment.zip")) as z:
    z.extractall(WORK)

PACK = f"{WORK}/candidate_pack"
DATA = f"{PACK}/data"

# Copy scripts
for f in ("audit_clean.py", "augment.py"):
    shutil.copy2(find(f), WORK)

# Clean and augment training data
subprocess.run(
    ["python", "audit_clean.py",
     "--train", f"{DATA}/train.jsonl",
     "--dev", f"{DATA}/dev.jsonl",
     "--out_dir", f"{WORK}/clean"],
    cwd=WORK,
    check=True
)

subprocess.run(
    ["python", "augment.py",
     "--clean", "clean/train_clean.jsonl",
     "--out", "clean/train_aug.jsonl",
     "--n", "400",
     "--seed", "42"],
    cwd=WORK,
    check=True
)

shutil.copy2(
    f"{WORK}/clean/cleaning_log.csv",
    OUT
)

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_clean = load_jsonl(f"{WORK}/clean/train_clean.jsonl")
train_aug = load_jsonl(f"{WORK}/clean/train_aug.jsonl")

dev = load_jsonl(f"{DATA}/dev.jsonl")
test = load_jsonl(f"{DATA}/test_inputs.jsonl")

SYSTEM = dev[0]["messages"][0]["content"]

assert all(
    r["messages"][0]["content"] == SYSTEM
    for r in train_aug + dev + test
)

assert all(
    [m["role"] for m in r["messages"]] == ["system", "user"]
    for r in test
)

print(
    "Dataset counts:",
    len(train_clean),
    len(train_aug),
    len(dev),
    len(test)
)

assert (len(train_clean), len(train_aug)) == (1296, 1696)

print(pd.read_csv(f"{OUT}/cleaning_log.csv").action.value_counts())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment: Google Colab
Source folder: /content/drive/MyDrive/vima3ya
Project folder: /content/drive/MyDrive/vima3ya
FOUND: training_assessment.zip
FOUND: audit_clean.py
FOUND: augment.py
FOUND: experiments.jsonl
FOUND: experiments_kaggle.jsonl
Dataset counts: 1296 1696 200 400
action
fix     175
drop    149
Name: count, dtype: int64


## 2. Chat template and sequence lengths
No row may be truncated: TRL's default `max_length=1024` cuts from the right, i.e. it would remove the answer of long email threads.

In [4]:
from transformers import AutoTokenizer
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
def render(msgs, gen_prompt): return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gen_prompt)
def ntok(s): return len(tok(s, add_special_tokens=False)["input_ids"])
print(render(train_clean[0]["messages"], False))
def stats(name, xs):
    xs = np.array(xs); print(f"{name:22} n={len(xs):5d} p50={np.percentile(xs,50):5.0f} p95={np.percentile(xs,95):5.0f} max={xs.max():5d}")
L_train = [ntok(render(r["messages"], False)) for r in train_aug]
stats("train prompt+answer", L_train)
stats("train answer", [ntok(r["messages"][2]["content"]) for r in train_aug])
stats("dev prompt", [ntok(render(r["messages"][:2], True)) for r in dev])
stats("test prompt", [ntok(render(r["messages"][:2], True)) for r in test])
MAX_LEN = 2304
assert max(L_train) <= MAX_LEN

<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.<|im_end|>
<|im_start|>assistant
{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>

train prompt+answer    n= 1696 p50=  176 p95=  225 max= 2215
train answer           n= 1696 p50=   67 p95=   80 max=   83
dev prompt             n=  200 p50=  106 p95=  142 max= 2115
test prompt            n=  400 p50=  108 p95=  151 max= 2089


## 3. Inference procedure (as required) and scoring
Only system+user messages, the model's own chat template, **greedy** (`do_sample=False`; Qwen's default
generation_config enables sampling and a repetition penalty, so both are overridden), `max_new_tokens=200`,
no constraints, no post-processing except `.strip()`. Batched calls sort prompts by length and cap tokens per batch.

In [5]:
GEN_KW = dict(max_new_tokens=200, do_sample=False, temperature=None, top_p=None, top_k=None,
              repetition_penalty=1.0, pad_token_id=tok.pad_token_id)

@torch.no_grad()
def predict(model, rows, system=None, batch_size=16, token_budget=12000):
    tok.padding_side = "left"
    texts = [render([{"role": "system", "content": system or r["messages"][0]["content"]}, r["messages"][1]], True) for r in rows]
    lens = [ntok(t) for t in texts]
    order = sorted(range(len(rows)), key=lambda i: lens[i])
    outs, i = {}, 0
    while i < len(order):
        longest = lens[order[min(i + batch_size, len(order)) - 1]]
        idx = order[i:i + max(1, min(batch_size, token_budget // longest))]
        enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, **GEN_KW)
        for j, g in zip(idx, gen[:, enc["input_ids"].shape[1]:]):
            outs[rows[j]["id"]] = {"id": rows[j]["id"], "output": tok.decode(g, skip_special_tokens=True).strip()}
        i += len(idx)
    return [outs[r["id"]] for r in rows]

sys.path.insert(0, PACK); import score as S
def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
def official_score(preds, name):
    path = f"{OUT}/{name}.jsonl"; save_jsonl(preds, path)
    print(subprocess.run(["python", f"{PACK}/score.py", "--gold", f"{DATA}/dev.jsonl", "--pred", path],
                         capture_output=True, text=True).stdout)
    return S.score(dev, preds)["all"]

## 4. Baselines: base model without fine-tuning (dev)
Baseline 1 uses the fixed system prompt. Baseline 2 adds SCHEMA.md verbatim to the system prompt (reference only: not allowed at test time).

In [6]:
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0}); base.eval()
print("Qwen default generation_config:", base.generation_config)
t0 = time.time(); preds_b1 = predict(base, dev); print(f"baseline 1: {time.time()-t0:.0f}s")
b1 = official_score(preds_b1, "dev_baseline1_fixed_prompt")
RULES_SYSTEM = SYSTEM + "\n\n" + open(f"{PACK}/SCHEMA.md", encoding="utf-8").read()
t0 = time.time(); preds_b2 = predict(base, dev, system=RULES_SYSTEM, batch_size=8); print(f"baseline 2: {time.time()-t0:.0f}s")
b2 = official_score(preds_b2, "dev_baseline2_rules_prompt")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen default generation_config: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

baseline 1: 189s
group                            n   valid   exact  fields
all                            200    0.0%    0.0%    0.0%
category=account_access         19    0.0%    0.0%    0.0%
category=fraud                  15    0.0%    0.0%    0.0%
category=kyc                    24    0.0%    0.0%    0.0%
category=offers                 18    0.0%    0.0%    0.0%
category=other                  16    0.0%    0.0%    0.0%
category=payment_failed         53    0.0%    0.0%    0.0%
category=refund                 55    0.0%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.

In [7]:
# Diagnostic only (NOT a valid score): separate "wrong format" from "wrong content".
def shape(o):
    if o.startswith("```"): return "markdown code fence"
    try: return "valid JSON object" if isinstance(json.loads(o), dict) else "JSON, not an object"
    except Exception: return "prose / invalid JSON"
for name, preds in [("baseline 1", preds_b1), ("baseline 2", preds_b2)]:
    m = [{"id": p["id"], "output": (re.search(r"\{.*\}", p["output"], re.S) or [p["output"]])[0]} for p in preds]
    d = S.score(dev, m)["all"]
    print(name, dict(collections.Counter(shape(p["output"]) for p in preds)),
          f"| fences stripped (diagnostic): exact={d['exact_match']:.1%} fields={d['mean_field_acc']:.1%}")
    print("   ", {k: f"{v:.0%}" for k, v in d["fields"].items()})
print(preds_b1[0]["output"][:300])
del base; gc.collect(); torch.cuda.empty_cache()

baseline 1 {'markdown code fence': 200} | fences stripped (diagnostic): exact=0.0% fields=0.0%
    {'category': '0%', 'priority': '0%', 'amount_paise': '0%', 'txn_date': '0%', 'txn_id': '0%', 'channel': '0%', 'language': '0%', 'needs_human': '0%'}
baseline 2 {'markdown code fence': 200} | fences stripped (diagnostic): exact=0.5% fields=61.4%
    {'category': '67%', 'priority': '34%', 'amount_paise': '86%', 'txn_date': '71%', 'txn_id': '64%', 'channel': '75%', 'language': '64%', 'needs_human': '30%'}
```json
{
  "status": "New",
  "priority": "High",
  "category": "Card Activation",
  "description": "User needs assistance with activating and using a card. Previous ticket #54047 is open.",
  "resolution": "Pending",
  "resolution_notes": "User needs to follow the activation instructions provided 


## 5. Training setup
| Setting | Value | Why |
|---|---|---|
| Data | clean (1,296) + 400 augmented rows | E4: amounts rewritten near the Rs 5k / Rs 50k thresholds, labels recomputed by SCHEMA rules |
| Format | prompt = [system, user], completion = [assistant] | TRL computes loss on completion tokens only |
| LoRA | r=16, alpha=32, dropout 0.05, all 7 linear projections | 18.5M trainable params (cap 50M) |
| Precision | fp16 base, fp32 adapter weights, fp16 AMP | T4 has no bf16; the fp16 grad scaler needs fp32 trainable weights |
| Optimisation | lr 2e-4 cosine, 5% warmup, batch 4 x accum 4, 3 epochs, seed 42 | effective batch 16 |
| Memory | gradient checkpointing, max_length 2304 (no truncation) | longest row is ~2.2k tokens |

In [8]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
def to_pc(rows):
    return Dataset.from_list([{"prompt": r["messages"][:2], "completion": [r["messages"][2]]} for r in rows])

def build_trainer(train_rows, r=16, alpha=32, dropout=0.05, epochs=3, lr=2e-4, bs=4, ga=4, seed=42, quant4=False):
    if quant4:   # E2: QLoRA (NF4 base weights, double quantization, fp16 compute)
        from transformers import BitsAndBytesConfig
        from peft import prepare_model_for_kbit_training
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                 bnb_4bit_compute_dtype=torch.float16)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0})
        model.enable_input_require_grads()
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=TARGETS,
                                             bias="none", task_type="CAUSAL_LM"))
    steps = math.ceil(len(train_rows) / (bs * ga)) * epochs
    args = SFTConfig(output_dir="/tmp/ckpt", num_train_epochs=epochs, per_device_train_batch_size=bs,
                     gradient_accumulation_steps=ga, per_device_eval_batch_size=8, learning_rate=lr,
                     lr_scheduler_type="cosine", warmup_steps=max(1, round(0.05 * steps)), weight_decay=0.0,
                     fp16=True, bf16=False, gradient_checkpointing=True,
                     gradient_checkpointing_kwargs={"use_reentrant": False}, max_length=MAX_LEN, packing=False,
                     completion_only_loss=True, logging_steps=10, eval_strategy="epoch", save_strategy="no",
                     report_to="none", seed=seed, data_seed=seed)
    trainer = SFTTrainer(model=model, args=args, train_dataset=to_pc(train_rows), eval_dataset=to_pc(dev),
                         processing_class=tok)
    for p in trainer.model.parameters():            # adapters must be fp32 for the fp16 grad scaler
        if p.requires_grad and p.dtype != torch.float32: p.data = p.data.float()
    n = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    assert n <= 50_000_000
    print(f"trainable parameters: {n:,}")
    return trainer

## 6. Checks before training
1. loss is computed only on the JSON answer + `<|im_end|>`; 2. the trainer's prompt equals the inference prompt;
3. no truncation; 4. worst-case memory: forward+backward on the 4 longest rows **with gradient checkpointing on**, as in training.

In [9]:

# Build trainer and inspect training data
trainer = build_trainer(train_aug)
ds = trainer.train_dataset

# Verify training labels and prompt masking
ex = trainer.data_collator([ds[0]])
ids, labels = ex["input_ids"][0], ex["labels"][0]

trained = tok.decode(ids[labels != -100])
prompt_part = tok.decode(ids[labels == -100])

print("TRAINED ON:", repr(trained))

assert trained.startswith("{")
assert "<|im_end|>" in trained
assert "Received:" not in trained

assert prompt_part == render(
    train_aug[0]["messages"][:2], True
)

# Check tokenized sequence lengths
lens = np.array([len(x) for x in ds["input_ids"]])

assert lens.max() < MAX_LEN

print("tokenized max length:", lens.max())

# Enable gradient checkpointing
trainer.model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# Release unused GPU memory
gc.collect()
torch.cuda.empty_cache()

# Test only the longest sequence (instead of four)
idx = [int(np.argmax(lens))]

batch = {
    k: v.to("cuda")
    for k, v in trainer.data_collator([ds[i] for i in idx]).items()
    if k in ("input_ids", "attention_mask", "labels")
}

# Forward and backward memory test
trainer.model.train()
torch.cuda.reset_peak_memory_stats()

with torch.autocast("cuda", dtype=torch.float16):
    loss = trainer.model(**batch).loss

loss.backward()

trainer.model.zero_grad(set_to_none=True)

print(
    f"Test batch {list(batch['input_ids'].shape)}: "
    f"peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB"
)

# Cleanup
del batch, loss, ex
gc.collect()
torch.cuda.empty_cache()

print("GPU memory diagnostic completed successfully.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/1696 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1696 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1696 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1696 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

trainable parameters: 18,464,768
TRAINED ON: '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>\n'
tokenized max length: 2215
Test batch [1, 2215]: peak 6.91 GiB
GPU memory diagnostic completed successfully.


## 7. Train the final adapter (E4 configuration)

In [10]:
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()
trainer.train()
TRAIN_MIN = (time.time() - t0) / 60; TRAIN_PEAK = torch.cuda.max_memory_allocated() / 2**30
print(f"training: {TRAIN_MIN:.1f} min | peak GPU memory: {TRAIN_PEAK:.2f} GiB")
ADAPTER = f"{OUT}/adapter"
trainer.model.save_pretrained(ADAPTER)
json.dump(trainer.state.log_history, open(f"{OUT}/train_log.json", "w"))
print(sorted(os.listdir(ADAPTER)), f"{os.path.getsize(f'{ADAPTER}/adapter_model.safetensors')/2**20:.1f} MB")
trainer.accelerator.free_memory(); del trainer, ds; gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/2**30:.2f} GiB")

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.016417,0.013363,0.014387,0.995037,351194.000000
2,0.004482,0.005559,0.005153,0.998382,702388.000000
3,0.001584,0.003729,0.002297,0.999010,1053582.000000


training: 25.7 min | peak GPU memory: 5.90 GiB
['README.md', 'adapter_config.json', 'adapter_model.safetensors'] 70.5 MB
GPU memory after cleanup: 0.02 GiB


## 8. Load the saved adapter exactly as a grader would, and evaluate on dev
fp16 base + `PeftModel.from_pretrained` (not merged).

In [11]:
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base, ADAPTER); model.eval()
t0 = time.time(); preds_dev = predict(model, dev); print(f"dev generation: {time.time()-t0:.0f}s")
final = official_score(preds_dev, "dev_final")
json.dump({"name": "final (E4 config, this notebook)", "exact_match": final["exact_match"],
           "mean_field_acc": final["mean_field_acc"], "json_valid": final["json_valid"], "fields": final["fields"],
           "train_minutes": round(TRAIN_MIN, 1), "train_peak_gib": round(TRAIN_PEAK, 2), "versions": VERSIONS},
          open(f"{OUT}/final_result.json", "w"), indent=1)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

dev generation: 110s
group                            n   valid   exact  fields
all                            200  100.0%   93.5%   98.7%
category=account_access         19  100.0%  100.0%  100.0%
category=fraud                  15  100.0%   80.0%   95.0%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%  100.0%  100.0%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   90.6%   98.3%
category=refund                 55  100.0%   90.9%   98.2%

per-field accuracy (all rows)
  category        98.0%
  priority        95.5%
  amount_paise    99.5%
  txn_date        99.0%
  txn_id          99.5%
  channel        100.0%
  language       100.0%
  needs_human     98.0%



## 9. Error analysis (dev)

In [12]:
gold = {r["id"]: json.loads(r["messages"][2]["content"]) for r in dev}
text = {r["id"]: r["messages"][1]["content"] for r in dev}
def parse(o):
    try:
        x = json.loads(o); return x if isinstance(x, dict) else None
    except Exception: return None
def same(a, b): return a == b and type(a) is type(b)
pred = {p["id"]: parse(p["output"]) for p in preds_dev}
fails = {i: {k: (g[k], None if pred[i] is None else pred[i].get(k)) for k in g
             if pred[i] is None or not same(pred[i].get(k), g[k])} for i, g in gold.items()}
fails = {i: d for i, d in fails.items() if d}

print("invalid JSON outputs:", sum(v is None for v in pred.values()))
print("\nper-field accuracy (weakest first):")
print(pd.Series(final["fields"]).sort_values().map(lambda v: f"{v:.1%}").to_string())
cat_exact = pd.Series({c: np.mean([i not in fails for i in gold if gold[i]["category"] == c]) for c in sorted({g["category"] for g in gold.values()})})
print("\nexact match by category (weakest first):"); print(cat_exact.sort_values().map(lambda v: f"{v:.1%}").to_string())
lang_exact = {l: np.mean([i not in fails for i in gold if gold[i]["language"] == l]) for l in ("en", "hi-en")}
print("\nexact match by language:", {k: f"{v:.1%}" for k, v in lang_exact.items()})

def bucket(field): return [(i, d[field]) for i, d in fails.items() if field in d]
print("\nerrors by type:")
for label, field in [("date extraction", "txn_date"), ("amount conversion", "amount_paise"), ("priority", "priority"),
                     ("Hinglish detection (language)", "language"), ("txn_id extraction", "txn_id"),
                     ("category", "category"), ("channel", "channel"), ("needs_human", "needs_human")]:
    print(f"  {label:30} {len(bucket(field)):2d}  {bucket(field)[:6]}")

# priority errors: is the amount right but the threshold comparison wrong?
print("\npriority errors with the gold amount (Rs):")
for i, (g, p) in bucket("priority"):
    a = gold[i]["amount_paise"]
    print(f"  {i}: expected {g}, predicted {p}, category={gold[i]['category']}, amount={'-' if a is None else f'{a/100:,.2f}'}")

invalid JSON outputs: 0

per-field accuracy (weakest first):
priority         95.5%
category         98.0%
needs_human      98.0%
txn_date         99.0%
amount_paise     99.5%
txn_id           99.5%
channel         100.0%
language        100.0%

exact match by category (weakest first):
fraud              80.0%
payment_failed     90.6%
refund             90.9%
account_access    100.0%
offers            100.0%
kyc               100.0%
other             100.0%

exact match by language: {'en': '94.7%', 'hi-en': '89.8%'}

errors by type:
  date extraction                 2  [('dv-00006', ('2026-06-30', '2026-07-01')), ('dv-00090', ('2026-07-10', '2026-07-11'))]
  amount conversion               1  [('dv-00124', (None, 702676793.6))]
  priority                        9  [('dv-00010', ('P3', 'P1')), ('dv-00037', ('P1', 'P2')), ('dv-00056', ('P2', 'P1')), ('dv-00088', ('P3', 'P2')), ('dv-00093', ('P1', 'P3')), ('dv-00124', ('P3', 'P2'))]
  Hinglish detection (language)   0  []
  txn_id extract

In [13]:
print(len(fails), "failing dev tickets (field: (expected, predicted))\n")
for i, d in fails.items():
    print(i, d)
    print("    ticket:", text[i].split("\n\n", 1)[1][:350].replace("\n", " | "))
    print("    output:", next(p["output"] for p in preds_dev if p["id"] == i)[:250], "\n")

13 failing dev tickets (field: (expected, predicted))

dv-00006 {'txn_date': ('2026-06-30', '2026-07-01')}
    ticket: @NimbusPayCare The App shows a fraudulent payment made on my account. It was day before yesterday. You can check np 75453 79360. Amount: 1.3 lakh
    output: {"category": "fraud", "priority": "P1", "amount_paise": 13000000, "txn_date": "2026-07-01", "txn_id": "NP7545379360", "channel": null, "language": "en", "needs_human": true} 

dv-00010 {'category': ('payment_failed', 'fraud'), 'priority': ('P3', 'P1'), 'needs_human': (False, True)}
    ticket: Subject: Problem with NimbusPay |  | URGENT!! | I didn't see the money I added to my card. ID: NP0194178721 INR 2,570 was the value. Please look into it. |  | Regards, | Vikram Singh
    output: {"category": "fraud", "priority": "P1", "amount_paise": 257000, "txn_date": null, "txn_id": "NP0194178721", "channel": "card", "language": "en", "needs_human": true} 

dv-00037 {'priority': ('P1', 'P2'), 'needs_human': (True, False)}

## 10. Test predictions and reproducibility check
The final file is generated **one ticket at a time** (batch size 1, no padding), the plainest possible
`model.generate` call, so a grader regenerating any row reproduces it. As evidence that batching does not change
outputs, a batched run on the same rows is compared to it.

In [14]:
t0 = time.time(); preds_test = predict(model, test, batch_size=1); print(f"test generation (batch 1): {(time.time()-t0)/60:.1f} min")
assert [p["id"] for p in preds_test] == [r["id"] for r in test] and len(preds_test) == 400
save_jsonl(preds_test, f"{OUT}/predictions_test.jsonl")
print("valid JSON:", sum(parse(p["output"]) is not None for p in preds_test), "/ 400")

batched = {p["id"]: p["output"] for p in predict(model, test)}
diff = [p["id"] for p in preds_test if p["output"] != batched[p["id"]]]
print(f"batched vs batch-1 outputs identical for {400-len(diff)}/400 rows; differing ids: {diff[:10]}")

# What a grader does: reload the adapter from disk in a fresh model and regenerate a random sample.
del model, base; gc.collect(); torch.cuda.empty_cache()
base2 = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map={"": 0})
model2 = PeftModel.from_pretrained(base2, ADAPTER); model2.eval()
sample = random.Random(0).sample(test, 25)
regen = predict(model2, sample, batch_size=1)
filed = {p["id"]: p["output"] for p in load_jsonl(f"{OUT}/predictions_test.jsonl")}
mismatch = [p["id"] for p in regen if p["output"] != filed[p["id"]]]
print(f"reproducibility: {25-len(mismatch)}/25 regenerated rows match the file", mismatch)
assert not mismatch

test generation (batch 1): 34.4 min
valid JSON: 400 / 400
batched vs batch-1 outputs identical for 400/400 rows; differing ids: []


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

reproducibility: 25/25 regenerated rows match the file []


## 11. Experiment table (E0–E3 on Colab T4, E4 on Kaggle T4; loaded from the run logs)
The code for every experiment is in the next cell (not re-run here, to stay within the time limit).

In [15]:
logs = []
for name in ("experiments.jsonl", "experiments_kaggle.jsonl"):
    p = find(name)
    if p: logs += [json.loads(l) for l in open(p)]
keep = {"dev_E0_clean_lora16_r16_v2": "E0 reference: clean, fp16 LoRA, r=16",
        "dev_E1_raw_lora16_r16": "E1 raw data",
        "dev_E2_clean_qlora4_r16": "E2 4-bit QLoRA",
        "dev_E3a_clean_lora16_r8": "E3a rank 8",
        "dev_E3b_clean_lora16_r32": "E3b rank 32",
        "dev_E4_clean_aug400_lora16_r16": "E4 + 400 threshold-augmented rows (Kaggle)"}
rows = [{"experiment": keep[r["name"]], "dev exact %": round(100 * r["exact_match"], 1),
         "dev field acc %": round(100 * r["mean_field_acc"], 1), "priority %": round(100 * r["fields"]["priority"], 1),
         "peak GPU GiB": r.get("train_peak_gib"), "train min": r.get("train_minutes")}
        for r in logs if r["name"] in keep]
if not rows: print("experiment logs not found next to the inputs; only the final run is shown")
rows.append({"experiment": "Final (E4 config, this notebook run)", "dev exact %": round(100 * final["exact_match"], 1),
             "dev field acc %": round(100 * final["mean_field_acc"], 1), "priority %": round(100 * final["fields"]["priority"], 1),
             "peak GPU GiB": round(TRAIN_PEAK, 2), "train min": round(TRAIN_MIN, 1)})
table = pd.DataFrame(rows)
table

,experiment,dev exact %,dev field acc %,priority %,peak GPU GiB,train min
0,"E0 reference: clean, fp16 LoRA, r=16",92.0,98.5,95.0,9.47,23.3
1,E1 raw data,85.0,97.1,87.5,5.90,25.0
2,E2 4-bit QLoRA,90.0,98.2,92.5,5.54,28.3
3,E3a rank 8,88.5,98.0,93.0,5.76,22.6
4,E3b rank 32,93.5,98.6,95.5,6.23,22.8
5,E4 + 400 threshold-augmented rows (Kaggle),93.0,98.8,96.0,5.90,26.5
6,"Final (E4 config, this notebook run)",93.5,98.7,95.5,5.90,25.7


In [16]:
RUN_ALL_EXPERIMENTS = False   # ~2.5 GPU hours; results above come from the saved logs
EXPERIMENTS = {
    "E0_clean_lora16_r16":  dict(data="clean"),
    "E1_raw_lora16_r16":    dict(data="raw"),                 # train_raw.jsonl written by audit_clean.py
    "E2_clean_qlora4_r16":  dict(data="clean", quant4=True),  # NF4, double quant, fp16 compute
    "E3a_clean_lora16_r8":  dict(data="clean", r=8, alpha=16),
    "E3b_clean_lora16_r32": dict(data="clean", r=32, alpha=64),
    "E4_clean_aug400_lora16_r16": dict(data="aug"),
}
if RUN_ALL_EXPERIMENTS:
    data = {"clean": train_clean, "raw": load_jsonl(f"{WORK}/clean/train_raw.jsonl"), "aug": train_aug}
    for name, cfg in EXPERIMENTS.items():
        cfg = dict(cfg); rows = data[cfg.pop("data")]
        t = build_trainer(rows, **cfg); torch.cuda.reset_peak_memory_stats(); t0 = time.time(); t.train()
        print(name, f"{(time.time()-t0)/60:.1f} min", f"{torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
        t.model.eval(); t.model.config.use_cache = True
        print(S.score(dev, predict(t.model, dev))["all"])
        t.accelerator.free_memory(); del t; gc.collect(); torch.cuda.empty_cache()

## 12. Files to submit

In [17]:
print(f"Saved in {OUT}:")
for f in sorted(os.listdir(OUT)): print("  ", f)
print("\nCopy to the repo: final/adapter/ -> adapter/, predictions_test.jsonl, cleaning_log.csv")

Saved in /content/drive/MyDrive/vima3ya/final:
   adapter
   cleaning_log.csv
   dev_baseline1_fixed_prompt.jsonl
   dev_baseline2_rules_prompt.jsonl
   dev_final.jsonl
   final_result.json
   predictions_test.jsonl
   train_log.json

Copy to the repo: final/adapter/ -> adapter/, predictions_test.jsonl, cleaning_log.csv
